In [35]:
import coremltools as ct
import torch
from transformers import AutoTokenizer, LlamaForCausalLM
import numpy as np
from src.model import KvCacheStateLlamaForCausalLM
from src.converter import LlamaCoreMLConverter


In [17]:
model_id = "meta-llama/Llama-3.2-1B-instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = LlamaForCausalLM.from_pretrained(model_id)
    

In [22]:
model = KvCacheStateLlamaForCausalLM(
    model_id,
    batch_size=1,
    context_size=2048
)

In [25]:
converter = LlamaCoreMLConverter(
        model,
        batch_size=1,
        context_size=2048
    )

In [27]:
mlmodel = converter.convert(quantize=False)
mlmodel.save("./models/llama-3.2-1b-instruct.mlpackage")

/Users/danielnewman/.virtualenvs/avi_venv/lib/python3.11/site-packages/transformers/modeling_utils.py:5006: FutureWarning: `_is_quantized_training_enabled` is going to be deprecated in transformers 4.39.0. Please use `model.hf_quantizer.is_trainable` instead
  warnings.warn(
`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.
The `seen_tokens` attribute is deprecated and will be removed in v4.41. Use the `cache_position` model input instead.
Torch var valueCache is added again.
Torch var keyCache is added again.
Running MIL default pipeline:  69%|██████▉   | 61/88 [00:35<01:01,  2.27s/ passes]/Users/danielnewman/.virtualenvs/avi_venv/lib/python3.11/site-packages/coremltools/converters/mil/mil/passes/defs/optimize_repeat_ops.py:433: RuntimeWarning: overflow encountered in cast
  max(cur_range.low, tmp_range.low), min(cur_range.high, tmp_range.high)
Running MIL backend_mlprogram pipeline: 100%|██████████| 12/12 [00:00<00:00, 59.17 pass

In [29]:
model = ct.models.MLModel('./models/llama-3.2-1b-instruct.mlpackage')

In [36]:
def _sample_token(logits, temperature):
    """
    Sample next token from logits using temperature
    """
    if temperature == 0:
        return int(np.argmax(logits))
        
    probs = np.exp(logits / temperature)
    probs = probs / np.sum(probs)
    return int(np.random.choice(len(probs), p=probs))

In [52]:
prompt = "Hello, how are you?"
inputs = tokenizer(prompt, return_tensors="np", padding=False)
input_ids = inputs["input_ids"]




In [57]:
print(input_slice)
print(causal_mask)

[[128000]]
[[1.]]


array([[128000,   9906,     11,   1268,    527,    499,     30]])